# 04 · Feature engineering — first batch

## Imports

In [ ]:
import json
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Load batch

In [ ]:
DATA = Path('../data')
GROUP = int(os.environ['DATA_API_GROUP'])
files = sorted(DATA.glob(f'group-{GROUP}-*.json'))
if not files:
    raise FileNotFoundError('Collect a batch in notebook 01 first.')
with files[0].open() as file:
    batch = json.load(file)
df = pd.DataFrame(batch['data'])
print(f"Group {GROUP} | batch {batch['batch_number']} | {len(df):,} rows")


## Create features

In [ ]:
features = df.copy()
features['total_rooms'] = features['bed'] + features['bath']
features['size_per_room'] = features['house_size'] / features['total_rooms'].where(features['total_rooms'] > 0)
features['previous_sale_year'] = pd.to_datetime(features['prev_sold_date'], errors='coerce').dt.year
features['house_size_squared'] = features['house_size'] ** 2
columns = ['bed', 'bath', 'house_size', 'total_rooms', 'size_per_room', 'previous_sale_year', 'house_size_squared']
display(features[columns].head())
display(features[columns].describe(percentiles=[.01, .5, .99]).T)


## Check relationships

In [ ]:
display(features[['price', 'total_rooms', 'size_per_room']].corr())
plot = features[['price', 'size_per_room']].dropna().sample(n=min(3000, len(features[['price', 'size_per_room']].dropna())), random_state=42)
sns.scatterplot(data=plot, x='size_per_room', y='price', alpha=.3)
plt.title('Size per room and price — sample')
plt.show()


Features use input values only. Squared terms are tested in the model pipeline; verify the sale date is known at prediction time.


## Notes

The shared feature pipeline uses room count and house size per room. See notebook 05 for the model comparison.
